In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# GROW control transfer: fixed one-pulse diagnosis

Select a GPU runtime and Run all. This new diagnostic fixes dev_p0_s0 and
zero-based index10. It does not overwrite or rerun the original full GROW
experiment, use old run inputs, scan strengths, or change the DCT target/loss.
All output, source archive and launcher logs use the independent directory
`MyDrive/Video-WM/GROWControlTransfer/grow_control_transfer_<UTC>`.

OFF prefix0..9 produces a saved common latent and complete native UniPC history.
One shared CFG5 velocity at10 supplies OFF/A/B branches. A/B each receive one
original predicted-clean DCT pulse (target amplitude .5, eta .1). Actual native
UniPC steps measure transmission; two CPU scalar native solver probes independently
predict its affine coefficient K. No Euler replacement is used.

Phase one costs22 Transformer forwards,13 native scheduler steps and2 local
gradients, plus2 separately counted scalar solver probes. Finite-state, snapshot,
history and numerical consistency checks use fixed tolerances. If valid, Run all
automatically continues11..49 with no further control and each arm's own poststep
history. Weak, zero or negative response, loss and bit recovery never stop the run.
The entire plan is256 TF,130 native steps,2 scalar probes,2 local gradients,
zero VAE/MP4 calls and no Transformer backward. Fixed3-arm/46-slice failures and
AB indices10..50 are retained, including setup failures.

Saved observations include independent K versus descriptive Khat, actual latent
and dtype-cast model-input differences, A/B versus OFF and A-B DCT/clean differences,
and terminal184 sign votes per bit with BER/erasures. The local loss ratio .81 is
the algebraic factor (1-.1)^2, not a retention rate. Numerical compatibility does
not prove adequate response, recovery, quality or generalization; the diagnostic
cannot alone explain the original20-control run's failure.

The reused environment restores torch2.11.0+cu128 and diffusers0.40.0 if needed,
then verifies them in a fresh subprocess. Matching torch is not reinstalled.
No GPU model-name gate. Local validation used CPU/Fake checks with exact official
0.40 scheduler source loaded in a0.39 host, not a complete0.40 model environment.
The assistant has not run pretrained models, GPU, Colab or Drive experiments.
This notebook has static and mocked-launcher validation only.

Source SHA: a567cb1420b5cb1422ecd992e198ac1faa1ef7af.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = 'a567cb1420b5cb1422ecd992e198ac1faa1ef7af'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'grow_control_transfer_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/GROWControlTransfer') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.grow_control_transfer_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status': result['status'], 'arm_denominator': result['arm_denominator'], 'numerical_gate_pass': result.get('numerical_gate_pass'), 'fixed_calls': result['fixed_calls'], 'actual_calls': result['actual_calls'], 'stages': result['stages'], 'arms': {k: v['status'] for k, v in result['arms'].items()}}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
